In [ ]:
import tensorflow_addons as tfa 
import keras
import numpy as np
encoder_inputs = keras.layers.Input(shape=[None], dtype=np.int32) 
decoder_inputs = keras.layers.Input(shape=[None], dtype=np.int32) 
sequence_lengths = keras.layers.Input(shape=[], dtype=np.int32) 
embeddings = keras.layers.Embedding(vocab_size, embed_size) 
encoder_embeddings = embeddings(encoder_inputs) 
decoder_embeddings = embeddings(decoder_inputs) 
encoder = keras.layers.LSTM(512, return_state=True) 
encoder_outputs, state_h, state_c = encoder(encoder_embeddings) 
encoder_state = [state_h, state_c] 
sampler = tfa.seq2seq.sampler.TrainingSampler() 
decoder_cell = keras.layers.LSTMCell(512) 
output_layer = keras.layers.Dense(vocab_size) 
decoder = tfa.seq2seq.basic_decoder.BasicDecoder(decoder_cell, sampler, 
                                                 output_layer=output_layer) 
final_outputs, final_state, final_sequence_lengths = decoder( 
    decoder_embeddings, initial_state=encoder_state, 
    sequence_length=sequence_lengths) 
Y_proba = tf.nn.softmax(final_outputs.rnn_output) 
model = keras.Model(inputs=[encoder_inputs, decoder_inputs, sequence_lengths], 
                    outputs=[Y_proba]) 

In [ ]:
#双向RNN
keras.layers.Bidirectional(keras.layers.GRU(10, return_sequences=True)) 

#集束搜索
beam_width = 10 
decoder = tfa.seq2seq.beam_search_decoder.BeamSearchDecoder( 
    cell=decoder_cell, beam_width=beam_width, output_layer=output_layer) 
decoder_initial_state = tfa.seq2seq.beam_search_decoder.tile_batch( 
    encoder_state, multiplier=beam_width) 
outputs, _, _ = decoder( 
    embedding_decoder, start_tokens=start_tokens, end_token=end_token, 
    initial_state=decoder_initial_state)